##Transform Circuits Data

1. Read bronze circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId->circuit_id, circuitName->circuit_name)
4. Rename columns to make them more meaningful (lat->latitude, long->longitude)
5. Filter out rows where circuit_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver circuits table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

####Step 1: Read the bronze circuits table

In [0]:
circuits_df = spark.read.table(bronze_table)

In [0]:
display(circuits_df)

circuitId,url,circuitName,lat,long,locality,country,ingestion_timestamp,source_file
null,https://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
null,https://en.wikipedia.org/wiki/Lusail_International_Circuit,losail international circuit,25.49,51.4542,lusail,Qatar,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,https://en.wikipedia.org/wiki/Adelaide_Street_Circuit,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,https://en.wikipedia.org/wiki/Ain-Diab_Circuit,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit,aintree,53.4769,-2.94056,liverpool,UK,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,https://en.wikipedia.org/wiki/Circuit_of_the_Americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,https://en.wikipedia.org/wiki/Anderstorp_Raceway,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,https://en.wikipedia.org/wiki/AVUS,avus,52.4806,13.2514,berlin,Germany,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,https://en.wikipedia.org/wiki/Bahrain_International_Circuit,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


####Step 2: Keep only the coluumns required for analysis (Drop url column)

There are 2 ways to do this... one by selecting the required columns and the other by dropping the unwanted columns

In [0]:
##This method uses selection of the required columns
from pyspark.sql import functions as F

circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
#This is the alternate version of the above method without using functions

# circuits_selected_df = circuits_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "ingestion_timestamp",
#     "source_file"
# )

####Step 3,4: Standardizing Column names

 - Standardize column names using snake_case
 - Rename column names to make them meaningful

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"})
)

#The above method only works for spark 3.4 version and above.
#For spark 3.3 and below, use the below method
# circuits_renamed_df = (
#     circuits_selected_df
#         .withColumnRenamed("circuitId", "circuit_id")
#         .withColumnRenamed("circuitName", "circuit_name")
#         .withColumnRenamed("lat", "latitude")
#         .withColumnRenamed("long", "longitude")
# )

In [0]:
display(circuits_renamed_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
null,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
null,losail international circuit,25.49,51.4542,lusail,Qatar,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,avus,52.4806,13.2514,berlin,Germany,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


####Step 5: Filter out rows where circuit_id is null (business key validation)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
display(circuits_valid_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,avus,52.4806,13.2514,berlin,Germany,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
baku,baku city circuit,40.3725,49.8533,baku,Azerbaijan,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
boavista,circuito da boavista,41.1705,-8.67325,oporto,Portugal,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


####Step 6: Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"]) # here we are dropping the duplicates based on the column circuit_id... if we use distinct(), it will dropthe duplicates w.r.t all columns

display(circuits_distinct_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,avus,52.4806,13.2514,berlin,Germany,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
baku,baku city circuit,40.3725,49.8533,baku,Azerbaijan,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
boavista,circuito da boavista,41.1705,-8.67325,oporto,Portugal,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


####Step 7: Transform the values of circuit_name and locality to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
        .withColumn("locality", F.initcap(F.col("locality")))
)

display(circuits_final_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,Ain Diab,33.5786,-7.6875,Casablanca,Morocco,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,Aintree,53.4769,-2.94056,Liverpool,UK,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,Circuit Of The Americas,30.1328,-97.6411,Austin,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,Scandinavian Raceway,57.2653,13.6042,Anderstorp,Sweden,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,Avus,52.4806,13.2514,Berlin,Germany,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
baku,Baku City Circuit,40.3725,49.8533,Baku,Azerbaijan,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
boavista,Circuito Da Boavista,41.1705,-8.67325,Oporto,Portugal,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


####Step 8: Write the transformed data to the silver 'circuits' table

In [0]:
(
    circuits_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
detroit,Detroit Street Circuit,42.3298,-83.0401,Detroit,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
mosport,Mosport International Raceway,44.0481,-78.6756,Ontario,Canada,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
red_bull_ring,Red Bull Ring,47.2197,14.7647,Spielberg,Austria,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
sebring,Sebring International Raceway,27.4547,-81.3483,Florida,USA,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
buddh,Buddh International Circuit,28.3487,77.5331,Uttar Pradesh,India,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
monaco,Circuit De Monaco,43.7347,7.42056,Monte Carlo,Monaco,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
pedralbes,Circuit De Pedralbes,41.3903,2.11667,Barcelona,Spain,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
tremblant,Circuit Mont-tremblant,46.1877,-74.6099,Quebec,Canada,2026-07-06T10:55:40.004Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
